# Step 1: Sample First 20 Tasks

From `first-20-task-sampling-strategy.md`:
- HotpotQA dev: random 10
- 2WikiMultiHopQA dev: random 10
- No cherry-picking, only filter malformed items
- Goal: test taxonomy stability, not maximize coverage

In [1]:
!pip install -q datasets

In [2]:
import random
from datasets import load_dataset

SEED = 42
random.seed(SEED)

## 1. Load HotpotQA dev set

In [3]:
hotpot = load_dataset("hotpot_qa", "fullwiki", split="validation")
print(f"HotpotQA dev size: {len(hotpot)}")
print(f"Columns: {hotpot.column_names}")
print("---")
print(hotpot[0])

README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7405 [00:00<?, ? examples/s]

HotpotQA dev size: 7405
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']
---
{'id': '5a8b57f25542995d1e6f1371', 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?', 'answer': 'yes', 'type': 'comparison', 'level': 'hard', 'supporting_facts': {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}, 'context': {'title': ['Adam Collis', 'Ed Wood (film)', 'Tyler Bates', 'Doctor Strange (2016 film)', 'Hellraiser: Inferno', 'Sinister (film)', 'Deliver Us from Evil (2014 film)', 'Woodson, Arkansas', 'Conrad Brooks', 'The Exorcism of Emily Rose'], 'sentences': [['Adam Collis is an American filmmaker and actor.', ' He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.', ' He also studied cinema at the University of Southern California from 1991 to 1997.', ' Collis first work was the assistant director for the Scott Derrickson\'s short "Love in the Ruins" (1995).', ' In 199

## 2. Load 2WikiMultiHopQA dev set

In [4]:
wiki2 = load_dataset("scholarly-shadows-syndicate/2WikiMultiHopQA", split="validation")
print(f"2WikiMultiHopQA dev size: {len(wiki2)}")
print(f"Columns: {wiki2.column_names}")
print("---")
print(wiki2[0])

README.md:   0%|          | 0.00/251 [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/343M [00:00<?, ?B/s]

dev.parquet:   0%|          | 0.00/30.1M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/28.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

2WikiMultiHopQA dev size: 12576
Columns: ['_id', 'type', 'question', 'context', 'supporting_facts', 'evidences', 'answer']
---
{'_id': '8813f87c0bdd11eba7f7acde48001122', 'type': 'compositional', 'question': 'Who is the mother of the director of film Polish-Russian War (Film)?', 'context': '[["Maheen Khan", ["Maheen Khan is a Pakistani fashion and costume designer, also an award winner fashion designer for fashion labels like\\" The Embroidery HouseMaheen\\" and\\" Gulabo\\".", "She has done many national and international fashion events and shows.", "She undertook embroidery for the film Snow White and the Huntsman and television series", "The Jewel in the Crown."]], ["Viktor Yeliseyev", ["Viktor Petrovich Yeliseyev( born June 9, 1950) is a Russian general, orchestra conductor and music teacher.", "He is the director of the Ministry of the Interior Ensemble, one of the two Russian Red Army Choirs."]], ["Alice Washburn", ["Alice Washburn( 1860- 1929) was an American stage and film actr

## 3. Random sample 10 + 10

Simple random sampling. Only filter obviously malformed items (empty question or empty answer).

In [5]:
# --- HotpotQA sampling ---
hp_indices = list(range(len(hotpot)))
random.shuffle(hp_indices)

hp_samples = []
for idx in hp_indices:
    item = hotpot[idx]
    q = item.get("question", "") or ""
    a = item.get("answer", "") or ""
    if len(q.strip()) < 10 or len(a.strip()) == 0:
        continue  # skip malformed
    hp_samples.append({
        "task_id": f"hp_dev_{idx:04d}",
        "dataset": "HotpotQA",
        "original_index": idx,
        "question": q.strip(),
        "answer": a.strip(),
        "type": item.get("type", ""),
        "level": item.get("level", ""),
    })
    if len(hp_samples) == 10:
        break

print(f"HotpotQA sampled: {len(hp_samples)}")

HotpotQA sampled: 10


In [6]:
# --- 2WikiMultiHopQA sampling ---
w2_indices = list(range(len(wiki2)))
random.shuffle(w2_indices)

w2_samples = []
for idx in w2_indices:
    item = wiki2[idx]
    q = item.get("question", "") or ""
    a = item.get("answer", "") or ""
    if len(q.strip()) < 10 or len(a.strip()) == 0:
        continue  # skip malformed
    w2_samples.append({
        "task_id": f"wiki_dev_{idx:04d}",
        "dataset": "2WikiMultiHopQA",
        "original_index": idx,
        "question": q.strip(),
        "answer": a.strip(),
        "type": item.get("type", ""),
    })
    if len(w2_samples) == 10:
        break

print(f"2WikiMultiHopQA sampled: {len(w2_samples)}")

2WikiMultiHopQA sampled: 10


## 4. Review sampled tasks

Look through all 20 tasks. Check:
- Is the question readable?
- Is the answer non-empty?
- Is it actually multi-hop?

Do NOT filter based on difficulty or "typicality".

In [7]:
all_samples = hp_samples + w2_samples

for i, s in enumerate(all_samples):
    print(f"[{i+1:02d}] {s['task_id']}  ({s['dataset']})")
    print(f"     Q: {s['question']}")
    print(f"     A: {s['answer']}")
    if s.get('type'):
        print(f"     type: {s['type']}")
    print()

[01] hp_dev_1547  (HotpotQA)
     Q: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
     A: flotilla
     type: bridge

[02] hp_dev_2054  (HotpotQA)
     Q: When did the park at which Tivolis Koncertsal is located open?
     A: 15 August 1843
     type: bridge

[03] hp_dev_0478  (HotpotQA)
     Q: What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
     A: Armenian
     type: comparison

[04] hp_dev_3245  (HotpotQA)
     Q: The school in which the Wilmslow Show is held is designated as what?
     A: Centre of Excellence
     type: bridge

[05] hp_dev_1119  (HotpotQA)
     Q: Who will Billy Howle be seen opposite in the upcoming British drama film directed by Dominic Cooke?
     A: Saoirse Ronan
     type: bridge

[06] hp_dev_3575  (HotpotQA)
     Q: What animated movie, starring Danny Devito, featured music written and produced by Kool Kojak?
     A: The Lorax
     type: br

## 5. Export to taxonomy.csv

Output matches the schema in `csv-field-examples.md`.

`reasoning_label` and `keep_drop` left blank — you fill these in during the annotation step.

In [8]:
import csv
import io

output = io.StringIO()
writer = csv.writer(output)
writer.writerow(["task_id", "dataset", "question", "answer", "reasoning_label", "keep_drop", "note"])

for s in all_samples:
    writer.writerow([
        s["task_id"],
        s["dataset"],
        s["question"],
        s["answer"],
        "",  # reasoning_label: fill during annotation
        "",  # keep_drop: fill during annotation
        "",  # note: fill during annotation
    ])

csv_text = output.getvalue()
print(csv_text)

task_id,dataset,question,answer,reasoning_label,keep_drop,note
hp_dev_1547,HotpotQA,What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?,flotilla,,,
hp_dev_2054,HotpotQA,When did the park at which Tivolis Koncertsal is located open?,15 August 1843,,,
hp_dev_0478,HotpotQA,What is the shared country of ancestry between Art Laboe and Scout Tufankjian?,Armenian,,,
hp_dev_3245,HotpotQA,The school in which the Wilmslow Show is held is designated as what?,Centre of Excellence,,,
hp_dev_1119,HotpotQA,Who will Billy Howle be seen opposite in the upcoming British drama film directed by Dominic Cooke?,Saoirse Ronan,,,
hp_dev_3575,HotpotQA,"What animated movie, starring Danny Devito, featured music written and produced by Kool Kojak?",The Lorax,,,
hp_dev_0978,HotpotQA,"Out of the actors who have played the role of Luc Deveraux in the Universal Soldier franchise, which actor has also starred in the movies Holby City, D

In [9]:
# Save to file (download from Colab afterwards)
with open("taxonomy_round1_raw.csv", "w", newline="") as f:
    f.write(csv_text)

print("Saved: taxonomy_round1_raw.csv")
print("Download this file, then fill reasoning_label / keep_drop / note columns.")

Saved: taxonomy_round1_raw.csv
Download this file, then fill reasoning_label / keep_drop / note columns.


## 6. Also save the full context (for later source-set construction)

Save supporting paragraphs / evidence so you can inspect reasoning structure during annotation.

In [10]:
import json

detailed = []
for s in all_samples:
    idx = s["original_index"]
    if s["dataset"] == "HotpotQA":
        item = hotpot[idx]
    else:
        item = wiki2[idx]
    detailed.append({
        "task_id": s["task_id"],
        "dataset": s["dataset"],
        "question": s["question"],
        "answer": s["answer"],
        "raw": {k: v for k, v in item.items()},
    })

with open("sampled_20_full.json", "w") as f:
    json.dump(detailed, f, indent=2, ensure_ascii=False)

print("Saved: sampled_20_full.json")
print("This file contains supporting paragraphs for annotation.")

Saved: sampled_20_full.json
This file contains supporting paragraphs for annotation.


---

## Next steps

1. Download `taxonomy_round1_raw.csv` and `sampled_20_full.json`
2. Read each question + its supporting paragraphs
3. Fill `reasoning_label` (bridge / comparison / temporal / distractor-heavy) and `keep_drop`
4. Copy the annotated CSV back to `pilot/taxonomy.csv`